In [34]:
import re
from abc import ABC, abstractmethod
from mistralai import Mistral

In [35]:
class BaseLMJudge(ABC):

    def __init__(self, client = None, args: dict[str, dict] = None):
        self.args = args
        self.client = client(**args["client_args"])

    @abstractmethod
    def __call__(self, gen_args):
        pass

In [36]:
class MistralJudge(BaseLMJudge):

    def __init__(self, client=Mistral, args: dict[str, dict] = None):
        super().__init__(clinet=client, args=args)

    def __call__(self, gen_args: dict):
        return self._parse_score(self._call_mistral(**gen_args))
    
    def _call_mistral(
        self,
        model_name: str = "mistral-large-latest",
        system_prompt: str = None,
        user_prompt: str = None,
        assistant_prompt: str = None,
        gen_config: dict = None
    ) -> str:
    
        messages = []
        if system_prompt is not None:
            messages.append({
                "role": "system",
                "content": system_prompt
            })
            
        if assistant_prompt is not None:
            messages.append({
                "role": "assistant",
                "content": assistant_prompt
            })
            
        if user_prompt is not None:
            messages.append({
                "role": "user",
                "content": user_prompt
            });
    
        if gen_config is None:
            gen_config = {}
            
        chat_response = self.client.chat.complete(
            model=model_name,
            messages=messages,
            **gen_config
        )
    
        return chat_response.choices[0].message.content

    def _parse_score(self, raw_output: str) -> int:
        pattern = r'.*Оценка модели: (\d).*'
        score = re.match(pattern, repr(raw_output)).group(1)
        return int(score)

In [13]:
client_args = dict(
    api_key = open("/home/jovyan/work/alexander_workspace/mistral_api_key", "r").read().strip()
)

In [30]:
user_prompt = '''Тебе нужно оценить качество ответа на вопрос. Тебе даны: вопрос, ground truth ответ, который точно правильный, и сгенерированный ответ, которому ты должна поставить оценку  1 или 0. Оценка 1 означает, что ответ правильный, оценка 0 означает, что ответ неправильный. Нужно сравнить сгенерированный ответ с ground truth ответом, принимая во внимание и сам вопрос. Ground truth ответ и сгенерированный ответ могут быть разной длинны, содержать разное количество информации, поэтому учитывай то, дан ли ответ на вопросительное слово из вопроса: если в вопросе "кто", то в сгенерированном ответе должен быть назван какой-то человек, персонаж или животное; если в вопросе "сколько", то в ответе должно быть число.
    
Пример 1:
Вопрос: "На какое место Everybody поднимается в чарте Hot Dance Club Songs?"
Ground truth ответ: "На 3-е место Everybody поднимается в чарте Hot Dance Club Songs."
Сгенерированный ответ: "3-е место."
Оценка модели: 1

Пример 2:
Вопрос: "В какой области много многолюдных поселков?"
Ground truth ответ: "В области земель Колхиды."
Сгенерированный ответ: "В этой области много многолюдных поселков."
Оценка модели: 0

Пример 3:
Вопрос: "В скольких томах опубликована История Рима?"
Ground truth ответ: "История Рима опубликована в четырёх томах."
Сгенерированный ответ: "Ответ: В трех томах."
Оценка модели: 0

Пример 4:
Вопрос: "Какая голова была у Микеланджело?"
Ground truth ответ: "У Микеланджело была круглая голова."
Сгенерированный ответ: "Голова у Микеланджело была круглая, лоб квадратный, изрезанный морщинами, с сильно выраженными надбровными дугами."
Оценка модели: 1'''

In [31]:
gen_args = dict(
    model_name = "mistral-large-latest",
    system_prompt = "Ты AI-ассистент, который говорит на русском языке.",
    user_prompt=user_prompt,
    assistant_prompt = None,
    gen_config = {}
)

In [32]:
import json 

json.dump(gen_args, open("gen_args.json", "w"))

In [15]:
args = dict(
    client_args = client_args,
)
lmjudge = MistralJudge(Mistral, args)

In [19]:
output = lmjudge(gen_args)
output

1